# Phase 2: Full-Quality Training and Experiments (garden)

Phase 1 proved the pipeline with short 7k-step runs. This notebook runs gsplat's **full 30k-step schedule** and compares training strategies on **garden at 1/4 resolution**, using the same setup as the published 3DGS results.

| Question | How we answer it |
|---|---|
| Are we as good as the paper? | 30k steps, official poses, `images_4`, every 8th image held out, VGG LPIPS. Compare with the published 3DGS garden numbers |
| What does capping the Gaussian count cost? | **MCMC** strategy with a cap of 1M (and optionally 500k) vs the **default** strategy, which grows freely (millions of Gaussians) |
| What runs on a free T4? | Peak VRAM, training time, render speed and file size for every run |

**One 30k run per Colab session.** gsplat 1.5.3 can't resume an interrupted run, so each run must finish in one session. Every run also saves its **7k-step** scores and checkpoint, so a late disconnect still leaves a usable result. Finished runs are skipped on later sessions, so just change `RUN_NOW` and press Run all.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. **Keep the tab open** during training: an idle tab can be disconnected.

⏱️ On a T4: install 2-5 min (first session ever: +15 min to compile gsplat once), download ~1 min, **MCMC 1M ≈ 35 min** (measured), **MCMC 500k** somewhat less. The **default** strategy grows to over 5M Gaussians, slows to ~3 steps/s and runs out of memory on a T4 (F2.1), so run it only on an L4/A100.

## 1. GPU, Drive, settings

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, json, time, re, subprocess, shutil
SCENE = 'garden'

# Experiments: name -> trainer arguments. Add rows to grow the experiment matrix (README Phase 2).
EXPERIMENTS = {
    'mcmc_1M':   ['mcmc', '--strategy.cap-max', '1000000'],   # memory-capped, usually as good or better
    'default':   ['default'],                                # original 3DGS densification (ADC). ⚠️ runs out of memory on a T4 at ~12k steps (F2.1): needs an L4/A100
    'mcmc_500k': ['mcmc', '--strategy.cap-max', '500000'],    # smaller file, how much quality do we lose?
}
RUN_NOW = ['mcmc_1M']    # ⬅️ one per session. On a free T4: mcmc_1M, then mcmc_500k

STEPS   = 30_000         # gsplat's full schedule; scores and checkpoints are also saved at 7k
PROJECT = '/content/drive/MyDrive/3dgs-digital-twin'
WORK    = '/content/work'
FIG     = f'{PROJECT}/results/phase2_{SCENE}_figures'
for d in [f'{PROJECT}/results', f'{PROJECT}/exports', f'{PROJECT}/cache', WORK, FIG]:
    os.makedirs(d, exist_ok=True)
run_name = lambda exp: f'phase2_{SCENE}_1of4_{exp}_{STEPS // 1000}k'

## 2. Install gsplat

The same install as before (fixes F0.1-F0.5), with a **new fix for the 15-minute compile (F1.5)**.

The wheel build kept failing, so we now keep gsplat's compiled CUDA library itself (`gsplat_cuda.so`) on Drive:
- **First session:** compile once (~15 min), then copy the `.so` to `MyDrive/3dgs-digital-twin/cache/gsplat_jit/`.
- **Later sessions:** copy it back and load it directly, which takes seconds. A small patch to gsplat's loader looks in `/content/gsplat_prebuilt` first. If the file doesn't fit (for example after a Colab torch update), gsplat compiles as before and a new cache is saved under a new name.

In [ ]:
import importlib.util
MARKER = '/content/.gs_deps_ok'          # /content survives a kernel restart (not a VM reset)

if not os.path.exists(MARKER):
    import torch
    np_before = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                               capture_output=True, text=True).stdout.strip()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gsplat==1.5.3'], check=True)

    REPO = f'{WORK}/gsplat'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'v1.5.3',
                        'https://github.com/nerfstudio-project/gsplat.git', REPO], check=True)
    reqs = open(f'{REPO}/examples/requirements.txt').read()

    # F0.1 + F0.2: git deps (several glued onto one line) -> split, build without isolation
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja', 'setuptools', 'wheel'], check=True)
    if 'T4' in torch.cuda.get_device_name(0):
        os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'   # faster builds, T4 only
    for url in re.findall(r'git\+https://github\.com/[^@\s]+@[0-9a-f]{40}', reqs):
        print('building', url.split('/')[-1][:60])
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', url], check=True)

    # F0.3: drop git lines + numpy<2 pin, install the rest
    rest = [l for l in reqs.splitlines() if 'git+https' not in l and not l.lower().startswith('numpy')]
    open('/tmp/reqs.txt', 'w').write('\n'.join(rest))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/tmp/reqs.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'remotezip', 'plyfile', 'numpy>=2.1,<2.5'], check=True)

    # F0.5: pycolmap fork uses np.uint64(-1) -> OverflowError on numpy 2 (patch the file, don't import it)
    f = os.path.join(os.path.dirname(importlib.util.find_spec('pycolmap').origin), 'scene_manager.py')
    s = open(f).read()
    if 'np.uint64(-1)' in s:
        open(f, 'w').write(s.replace('np.uint64(-1)', 'np.uint64(18446744073709551615)')); print('patched pycolmap')

    # F0.4: make gsplat's examples/datasets a regular package (beats HF `datasets`)
    open(f'{REPO}/examples/datasets/__init__.py', 'a').close()

    open(MARKER, 'w').write('ok')
    np_after = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                              capture_output=True, text=True).stdout.strip()
    if np_before != np_after:
        print(f'numpy {np_before} -> {np_after}: restarting runtime. Then "Run all" again.')
        time.sleep(2); os.kill(os.getpid(), 9)

REPO = f'{WORK}/gsplat'
check = subprocess.run([sys.executable, '-c',
    'import numpy, gsplat, fused_ssim, pycolmap, tyro; print("numpy", numpy.__version__, "| gsplat", gsplat.__version__)'],
    capture_output=True, text=True, cwd=f'{REPO}/examples')
print(check.stdout or check.stderr[-2000:])

In [ ]:
# F1.5 (second fix): keep gsplat's compiled CUDA library on Drive and load it directly in later sessions
import torch, importlib.util
PREBUILT = '/content/gsplat_prebuilt'
cap = torch.cuda.get_device_capability()
KEY = f"gsplat1.5.3_torch{torch.__version__}_py{sys.version_info.major}{sys.version_info.minor}_sm{cap[0]}{cap[1]}"
JIT_CACHE = f'{PROJECT}/cache/gsplat_jit/{KEY}'

# 1. Patch gsplat's loader: use /content/gsplat_prebuilt/gsplat_cuda.so if it exists, otherwise behave as before
BACKEND = os.path.join(os.path.dirname(importlib.util.find_spec('gsplat').origin), 'cuda', '_backend.py')
src = open(BACKEND).read()
if 'GSPLAT_PREBUILT_DIR' not in src:
    old = '    from gsplat import csrc as _C\n'
    new = ('    _pre = os.environ.get("GSPLAT_PREBUILT_DIR", "/content/gsplat_prebuilt")\n'
           '    if os.path.exists(os.path.join(_pre, "gsplat_cuda.so")):\n'
           '        _C = _import_module_from_library("gsplat_cuda", _pre, True)\n'
           '    else:\n'
           '        from gsplat import csrc as _C\n')
    assert src.count(old) == 1, 'gsplat _backend.py changed, patch needs updating'
    open(BACKEND, 'w').write(src.replace(old, new)); print('patched gsplat loader')

# 2. Restore the cached library from Drive
os.makedirs(PREBUILT, exist_ok=True)
if not os.path.exists(f'{PREBUILT}/gsplat_cuda.so') and os.path.exists(f'{JIT_CACHE}/gsplat_cuda.so'):
    shutil.copy(f'{JIT_CACHE}/gsplat_cuda.so', PREBUILT)

def load_gsplat_cuda():
    t = time.time()
    r = subprocess.run([sys.executable, '-c', 'from gsplat.cuda._backend import _C; print("SO=", _C.__file__ if _C else None)'],
                       capture_output=True, text=True)
    so = re.search(r'SO= (.+)', r.stdout)
    return (so.group(1).strip() if so else None), time.time() - t, r

so, dt, r = load_gsplat_cuda()
if so and so.startswith(PREBUILT):
    print(f'✅ gsplat CUDA loaded from the Drive cache in {dt:.0f}s ({KEY})')
else:
    # 3. No usable cache: the import above compiled it (~15 min). Save the result for next time.
    if not so or so == 'None':
        print(r.stdout[-1500:], r.stderr[-3000:]); raise RuntimeError('gsplat CUDA failed to load or compile')
    print(f'compiled gsplat CUDA in {dt/60:.1f} min -> saving to Drive')
    os.makedirs(JIT_CACHE, exist_ok=True)
    shutil.copy(so, f'{JIT_CACHE}/gsplat_cuda.so'); shutil.copy(so, f'{PREBUILT}/gsplat_cuda.so')
    so2, dt2, _ = load_gsplat_cuda()
    print(f"{'✅' if so2 and so2.startswith(PREBUILT) else '❌'} reload from cache: {dt2:.0f}s ({so2})")

## 3. Download garden (official poses + 1/4-res images)

The published 3DGS results for outdoor Mip-NeRF 360 scenes use the provided `images_4` and the official poses, so we do the same. We train this folder with `--data_factor 1` (F1.2: with factor > 1 gsplat would shrink the images again).

In [ ]:
from remotezip import RemoteZip
URL = 'http://storage.googleapis.com/gresearch/refraw360/360_v2.zip'
DATA = f'{WORK}/data/360_v2/{SCENE}'
WANT = ('sparse/', 'poses_bounds.npy', 'images_4/')
if not os.path.exists(f'{DATA}/images_4'):
    t = time.time()
    with RemoteZip(URL) as z:
        members = [m for m in z.namelist() if f'{SCENE}/' in m and not m.endswith('/')
                   and any(m.split(f'{SCENE}/', 1)[1].startswith(w) for w in WANT)]
        for m in members:
            dst = f'{DATA}/' + m.split(f'{SCENE}/', 1)[1]
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            with z.open(m) as src_, open(dst, 'wb') as f:
                f.write(src_.read())
    print(f'downloaded {len(members)} files in {time.time()-t:.0f}s')
if not os.path.exists(f'{DATA}/images'):
    os.symlink(f'{DATA}/images_4', f'{DATA}/images')
import imageio.v2 as imageio
print(len(os.listdir(f'{DATA}/images')), 'images | size', imageio.imread(glob.glob(f'{DATA}/images/*')[0]).shape)

## 4. Train

Each run writes to `MyDrive/3dgs-digital-twin/results/<run>/` and evaluates at **7k and 30k** steps (VGG LPIPS). A run that already has its 30k scores is skipped. A run that stopped early is started again from zero, because gsplat can't resume.

In [ ]:
help_txt = subprocess.run([sys.executable, 'simple_trainer.py', 'default', '--help'], capture_output=True,
                          text=True, cwd=f'{REPO}/examples').stdout
assert re.search(r'lpips.net', help_txt), 'this gsplat has no --lpips_net option'

def step_of(path):
    return int(re.search(r'step(\d+)', os.path.basename(path)).group(1))

def stats_files(rd, stage):
    return sorted(glob.glob(f'{rd}/stats/{stage}_step*.json'), key=step_of)    # numeric order: 6999 before 29999

def check_resolution(rd, data_dir):
    expected = imageio.imread(sorted(glob.glob(f'{data_dir}/images/*'))[0]).shape[1]
    actual = imageio.imread(sorted(glob.glob(f'{rd}/renders/val_step*_*.png'))[0]).shape[1] / 2   # GT | render
    if abs(actual - expected) / expected > 0.1:
        raise RuntimeError(f'Trained at {actual:.0f}px instead of {expected}px: wrong data_factor? See F1.2')
    return int(actual)

def train(exp):
    rd = f'{PROJECT}/results/{run_name(exp)}'
    done = [f for f in stats_files(rd, 'val') if step_of(f) == STEPS - 1]
    if done:
        print(f'✔ {run_name(exp)}: already trained'); return
    if os.path.exists(rd):                      # a half-finished run can't be resumed: keep its files, start fresh
        shutil.move(rd, rd + time.strftime('_failed_%Y%m%d_%H%M'))
    cmd = [sys.executable, 'simple_trainer.py', *EXPERIMENTS[exp], '--data_dir', DATA, '--data_factor', '1',
           '--result_dir', rd, '--max_steps', str(STEPS), '--disable_viewer', '--lpips_net', 'vgg']
    os.makedirs(rd, exist_ok=True)
    print('$', ' '.join(cmd[1:]))
    t0 = last = time.time(); buf = ''; tail = []
    env = {**os.environ, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}   # less memory fragmentation (F2.1)
    p = subprocess.Popen(cmd, cwd=f'{REPO}/examples', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    with open(f'{rd}/train_log.txt', 'w') as lf:
        while (chunk := p.stdout.read(512)):
            lf.write(chunk); buf += chunk
            parts = re.split(r'[\r\n]', buf); buf = parts[-1]
            for ln in filter(str.strip, parts[:-1]):
                tail = (tail + [ln])[-40:]
                if 'PSNR' in ln or time.time() - last > 120:
                    print(f'[{(time.time()-t0)/60:5.1f} min] {ln.strip()[:150]}'); last = time.time()
    if p.wait() != 0:
        print('\n'.join(tail)); raise RuntimeError(f'training failed, log: {rd}/train_log.txt')
    json.dump({'wall_min': round((time.time() - t0) / 60, 1)}, open(f'{rd}/wall_time.json', 'w'))
    print(f'✔ {run_name(exp)} finished in {(time.time()-t0)/60:.1f} min | trained at width {check_resolution(rd, DATA)}px')

for exp in RUN_NOW:
    train(exp)

## 5. Results: every finished run vs the published 3DGS numbers

Reads all runs from Drive, including ones from earlier sessions.
- **Train time** is gsplat's own training clock (without evaluation). **Peak VRAM** is the most GPU memory PyTorch used.
- **Render FPS** comes from the evaluation pass at 1297×840, so it's a rough guide, not a viewer benchmark.
- **Raw `.ply` size** is computed from the Gaussian count (62 floats per Gaussian at SH degree 3), so nothing big is written to Drive.

**Published reference:** 3DGS (Kerbl et al. 2023), garden at 30k steps: PSNR 27.41, SSIM 0.868, LPIPS 0.103. Check these against the paper's per-scene table before quoting them. **Phase 2 target: within ~1 dB.**

In [ ]:
import numpy as np, csv, datetime
PUBLISHED = {'psnr': 27.41, 'ssim': 0.868, 'lpips': 0.103}
BYTES_PER_GS = 62 * 4

rows = []
for exp in EXPERIMENTS:
    rd = f'{PROJECT}/results/{run_name(exp)}'
    vals = {step_of(f): json.load(open(f)) for f in stats_files(rd, 'val')}
    trains = {step_of(f): json.load(open(f)) for f in stats_files(rd, 'train')}
    for step in sorted(vals):
        v, tr = vals[step], trains.get(step, {})
        rows.append({'exp': exp, 'steps': step + 1, 'psnr': v['psnr'], 'ssim': v['ssim'], 'lpips': v['lpips'],
                     'num_GS': v['num_GS'], 'train_min': tr.get('ellipse_time', np.nan) / 60, 'mem_gb': tr.get('mem', np.nan),
                     'fps': 1 / v['ellipse_time'], 'ply_mb': v['num_GS'] * BYTES_PER_GS / 1e6})
if not rows:
    print('No finished runs yet.')
else:
    print(f"{'experiment':>10} | {'steps':>6} | {'PSNR':>6} | {'SSIM':>6} | {'LPIPS':>6} | {'#Gaussians':>10} | "
          f"{'train':>7} | {'VRAM':>6} | {'FPS':>5} | {'.ply':>7} | vs paper")
    for r in rows:
        gap = r['psnr'] - PUBLISHED['psnr']
        verdict = (('✅' if gap > -1 else '❌') + f' {gap:+.2f} dB') if r['steps'] == STEPS else '(7k)'
        print(f"{r['exp']:>10} | {r['steps']:>6} | {r['psnr']:6.2f} | {r['ssim']:6.3f} | {r['lpips']:6.3f} | {r['num_GS']:>10,} | "
              f"{r['train_min']:5.1f} m | {r['mem_gb']:4.1f}GB | {r['fps']:5.1f} | {r['ply_mb']:5.0f}MB | {verdict}")
    print(f"{'paper':>10} | {STEPS:>6} | {PUBLISHED['psnr']:6.2f} | {PUBLISHED['ssim']:6.3f} | {PUBLISHED['lpips']:6.3f} |")
    json.dump(rows, open(f'{FIG}/phase2_results.json', 'w'), indent=1)

    # shared metrics.csv: one row per finished 30k run
    CSV = f'{PROJECT}/results/metrics.csv'
    header = next(csv.reader(open(CSV))) if os.path.exists(CSV) else \
        ['date', 'run', 'scene', 'res', 'strategy', 'steps', 'num_gaussians', 'psnr', 'ssim', 'lpips', 'gpu', 'mem_gb', 'ply_mb']
    final = [r for r in rows if r['steps'] == STEPS]
    names = {run_name(r['exp']) for r in final}
    old = [r for r in csv.DictReader(open(CSV)) if r['run'] not in names] if os.path.exists(CSV) else []
    with open(CSV, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=header, extrasaction='ignore'); w.writeheader(); w.writerows(old)
        for r in final:
            w.writerow({'date': str(datetime.date.today()), 'run': run_name(r['exp']), 'scene': SCENE, 'res': '1/4',
                        'strategy': ' '.join(EXPERIMENTS[r['exp']]), 'steps': STEPS, 'num_gaussians': r['num_GS'],
                        'psnr': round(r['psnr'], 3), 'ssim': round(r['ssim'], 4), 'lpips': f"{round(r['lpips'], 4)} (vgg)",
                        'gpu': torch.cuda.get_device_name(0), 'mem_gb': round(r['mem_gb'], 2), 'ply_mb': round(r['ply_mb'], 1)})
    print('\nlogged', len(final), 'finished run(s) to', CSV)

## 6. Render grids: real | render | error

For three fixed test views, the brighter the error map, the bigger the mistake. Saved to `results/phase2_garden_figures/` for the README.

In [ ]:
import matplotlib.pyplot as plt
finished = [e for e in EXPERIMENTS if any(r['exp'] == e and r['steps'] == STEPS for r in rows)]
VIEWS = [0, 8, 16]
for exp in finished:
    rd = f'{PROJECT}/results/{run_name(exp)}'
    fig, ax = plt.subplots(len(VIEWS), 3, figsize=(18, 4 * len(VIEWS)))
    for i, v in enumerate(VIEWS):
        im = imageio.imread(f'{rd}/renders/val_step{STEPS - 1}_{v:04d}.png')[..., :3].astype(np.float32) / 255
        w = im.shape[1] // 2; gt, pr = im[:, :w], im[:, w:]
        err = np.abs(gt - pr).mean(-1)
        psnr = -10 * np.log10(((gt - pr) ** 2).mean())
        ax[i, 0].imshow(gt); ax[i, 0].set_title(f'real (test view {v})')
        ax[i, 1].imshow(pr); ax[i, 1].set_title(f'{exp} render, PSNR {psnr:.2f}')
        ax[i, 2].imshow(err, cmap='inferno', vmin=0, vmax=0.25); ax[i, 2].set_title('error (black = exact)')
    for a in ax.flat: a.axis('off')
    plt.tight_layout(); plt.savefig(f'{FIG}/grid_{exp}.jpg', dpi=80); plt.show()

# same crop from every finished run, to compare fine detail
if len(finished) > 1:
    fig, ax = plt.subplots(1, len(finished) + 1, figsize=(5 * (len(finished) + 1), 5))
    for j, exp in enumerate(finished):
        im = imageio.imread(f'{PROJECT}/results/{run_name(exp)}/renders/val_step{STEPS - 1}_0000.png')[..., :3]
        w = im.shape[1] // 2; h = im.shape[0]
        crop = lambda x: x[h // 3: h // 3 + 300, w // 3: w // 3 + 300]
        if j == 0: ax[0].imshow(crop(im[:, :w])); ax[0].set_title('real')
        ax[j + 1].imshow(crop(im[:, w:])); ax[j + 1].set_title(exp)
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.savefig(f'{FIG}/crop_compare.jpg', dpi=90); plt.show()
print('figures in', FIG)

## 7. Export the run you want to view

Converts a checkpoint to a standard 3DGS `.ply` for SuperSplat. The default strategy can produce a file of over 1 GB, so start with `mcmc_1M` (~250 MB).

In [ ]:
from plyfile import PlyData, PlyElement
EXPORT = 'mcmc_1M'          # ⬅️ which finished run to export
DOWNLOAD = False            # True = also download the file to this computer (uses your internet)

def ckpt_to_ply(ckpt_path, ply_path):
    s = torch.load(ckpt_path, map_location='cpu', weights_only=False)['splats']
    means = s['means'].numpy()
    sh0 = s['sh0'].transpose(1, 2).flatten(1).numpy()            # [N,1,3] -> [N,3]
    shN = s['shN'].transpose(1, 2).flatten(1).numpy()            # [N,K,3] -> [N,3K] (channel-major, 3DGS convention)
    opac = s['opacities'].numpy()[:, None]                       # logit space
    scales = s['scales'].numpy()                                 # log space
    quats = s['quats'].numpy()                                   # wxyz
    normals = np.zeros_like(means)
    attrs = ['x','y','z','nx','ny','nz'] + [f'f_dc_{i}' for i in range(sh0.shape[1])] \
          + [f'f_rest_{i}' for i in range(shN.shape[1])] + ['opacity'] \
          + [f'scale_{i}' for i in range(3)] + [f'rot_{i}' for i in range(4)]
    data = np.concatenate([means, normals, sh0, shN, opac, scales, quats], axis=1).astype(np.float32)
    el = np.empty(data.shape[0], dtype=[(a, 'f4') for a in attrs])
    for i, a in enumerate(attrs):
        el[a] = data[:, i]
    PlyData([PlyElement.describe(el, 'vertex')]).write(ply_path)
    return data.shape[0]

rd = f'{PROJECT}/results/{run_name(EXPORT)}'
ckpts = sorted(glob.glob(f'{rd}/ckpts/ckpt_*_rank0.pt'), key=lambda p: int(re.search(r'ckpt_(\d+)', p).group(1)))
assert ckpts, f'no checkpoint for {EXPORT}: train it first'
PLY = f'{PROJECT}/exports/{run_name(EXPORT)}.ply'
n = ckpt_to_ply(ckpts[-1], PLY)
print(f'{n:,} Gaussians from {os.path.basename(ckpts[-1])} -> {PLY} ({os.path.getsize(PLY)/1e6:.0f} MB)')
if DOWNLOAD:
    from google.colab import files
    files.download(PLY)

## ✅ Phase 2 (garden) checklist
- [ ] `✅ gsplat CUDA loaded from the Drive cache` on the second session (F1.5 fixed)
- [ ] `mcmc_1M` 30k: within ~1 dB of the paper
- [ ] `default` 30k: within ~1 dB of the paper
- [ ] `mcmc_500k`: how much quality is lost for a file half the size
- [ ] Render grids saved; copy the best one into `docs/images/`
- [ ] The exported `.ply` looks right in SuperSplat
- [ ] Anything that broke → `docs/failures.md`

**Paste back into the chat:** the output of section 2's second cell (compile or cache line), section 4's last lines, and the section 5 table.

**Next sessions:** set `RUN_NOW = ['default']`, then `['mcmc_500k']`, and Run all. Finished runs are skipped.